# 🗺️ Notebook 7 — 7 Katmanlı Master Map
## AFETSONAR • Calamitas AI • Teknofest 2025 • Plan v2 Phase 7

---

## 🎯 Amaç

Notebook 5 (hasar analizi + priority) ve Notebook 6 (routing engine) çıktılarını birleştirip **7 katmanlı interaktif karar destek haritası** oluşturuyoruz.

### 7 Katman

| # | Katman | İçerik |
|---|---|---|
| 1 | **Base Tile** | OSM + Esri Uydu toggle |
| 2 | **🏠 Bina İşaretleri** | 27 bina, hasar rengi, popup (priority, survival, ETA) |
| 3 | **🔥 Hasar Yoğunluğu** | MarkerCluster + HeatMap (KDE) |
| 4 | **👥 Ekip Bölgeleri** | K-means Voronoi + 5 hastane |
| 5 | **🛣️ Yol Durumu** | Gradient edge weight (sadece hasarlı yollar) |
| 6 | **🗺️ Ekip Rotaları** | AntPath + alternatif rotalar |
| 7 | **🚁 Helikopter LZ** | Potansiyel iniş noktaları |

### Ek Özellikler

- LayerControl (aç/kapat)
- Fullscreen butonu
- MiniMap
- MeasureControl (mesafe ölçümü)
- AFETSONAR başlık overlay
- Renk açıklaması legend

---

> 💡 Bu harita **Teknofest jürisine** gösterilecek. Jüri binaya tıkladığında "bu gerçek bir afet karar destek sistemi" diyecek.


In [1]:
# ============================================================
# HÜCRE 2: Drive mount + paketler
# ============================================================
# Bu hücre ne yapıyor?
#   Drive bağlar, folium + plugins yükler
# Beklenen çıktı: "✅ folium X.Y"
# Hata olursa: pip install folium --quiet

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, os, sys
for pkg in ["folium", "geopandas", "branca"]:
    try: __import__(pkg)
    except ImportError: subprocess.run(["pip","install","-q",pkg], check=True)

import folium, folium.plugins, geopandas as gpd
import pandas as pd, numpy as np, json, pickle, math
print(f"✅ folium {folium.__version__}")

PROJECT = "/content/drive/MyDrive/AFETSONAR"
IN_NB5  = os.path.join(PROJECT, "outputs/notebook5")
IN_NB6  = os.path.join(PROJECT, "outputs/notebook6")
OUT_NB7 = os.path.join(PROJECT, "outputs/notebook7")
os.makedirs(OUT_NB7, exist_ok=True)


Mounted at /content/drive
✅ folium 0.20.0


## 📂 Notebook 5+6 Çıktılarını Yükle


In [2]:
# ============================================================
# HÜCRE 4: Tüm dosyaları yükle + sanity check
# ============================================================
# Bu hücre ne yapıyor?
#   NB5 + NB6 CSV/GeoJSON/pickle yükler
# Beklenen çıktı: "27 bina · 5 ekip · 11 alt rota"
# Hata olursa: "Notebook 5/6 çalıştır" mesajı

def _load(path, label):
    assert os.path.exists(path), f"❌ {label} yok: {path}\n   Önce ilgili notebook'u çalıştır."
    return path

# NB5
priority_df = pd.read_csv(_load(os.path.join(IN_NB5, "priority_test.csv"), "priority_test"))
buildings_geo_df = pd.read_csv(_load(os.path.join(IN_NB5, "buildings_with_geo.csv"), "buildings_with_geo"))
with open(_load(os.path.join(IN_NB5, "hospitals.geojson"), "hospitals.geojson")) as f:
    hospitals_geo = json.load(f)
with open(_load(os.path.join(IN_NB5, "teams.geojson"), "teams.geojson")) as f:
    teams_geo = json.load(f)
lz_path = os.path.join(IN_NB5, "lz_candidates.csv")
lz_df = pd.read_csv(lz_path) if os.path.exists(lz_path) else pd.DataFrame()

# NB6
master_df = pd.read_csv(_load(os.path.join(IN_NB6, "master_routing_table.csv"), "master_routing"))
with open(_load(os.path.join(IN_NB6, "team_routes.geojson"), "team_routes")) as f:
    team_routes_geo = json.load(f)
with open(_load(os.path.join(IN_NB6, "team_chains.json"), "team_chains")) as f:
    team_chains = json.load(f)
alt_path = os.path.join(IN_NB6, "alternative_routes.geojson")
with open(_load(alt_path, "alternative_routes")) as f:
    alt_routes_geo = json.load(f)
# Gradient graph
gw_path = os.path.join(IN_NB6, "gradient_weights.gpickle")
if os.path.exists(gw_path):
    with open(gw_path, "rb") as f:
        G = pickle.load(f)
    print(f"✓ Gradient graph: {G.number_of_edges()} edges")
else:
    G = None
    print("⚠️  gradient_weights.gpickle yok, yol durumu katmanı atlanacak")

# Merge priority_df + master_df for popup
popup_df = priority_df.merge(
    master_df[["building_id", "route_method", "segment_distance_m",
               "eta_vehicle_min", "alternative_count"]].drop_duplicates("building_id"),
    on="building_id", how="left"
)

HOSPITALS = [{"id": f["properties"]["id"], "name": f["properties"]["name"],
              "color": f["properties"]["color"],
              "lat": f["geometry"]["coordinates"][1],
              "lon": f["geometry"]["coordinates"][0]}
             for f in hospitals_geo["features"]]

TEAMS = [{"team_id": f["properties"]["team_id"],
          "color": f["properties"]["color"],
          "lat": f["geometry"]["coordinates"][1],
          "lon": f["geometry"]["coordinates"][0],
          "n_buildings": f["properties"]["n_buildings"]}
         for f in teams_geo["features"]]

print(f"\n{'='*50}")
print(f"📊 {len(priority_df)} bina · {len(HOSPITALS)} hastane · {len(TEAMS)} ekip")
print(f"   {len(alt_routes_geo['features'])} alternatif rota · {len(lz_df)} LZ adayı")
print(f"   route_method: {master_df['route_method'].value_counts().to_dict()}")


✓ Gradient graph: 1986 edges

📊 27 bina · 5 hastane · 5 ekip
   11 alternatif rota · 68 LZ adayı
   route_method: {'kara_yolu': 23, 'erisilemez_dusuk_oncelik': 1}


## 🗺️ KATMAN 1: Base Tile

OSM (default) + Esri Uydu toggle.


In [3]:
# ============================================================
# HÜCRE 6: Base map + 2 tile layer
# ============================================================
center_lat = priority_df["lat_center"].mean()
center_lon = priority_df["lon_center"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=16,
               tiles="OpenStreetMap", control_scale=True,
               width="100%", height="100%")

# Esri Uydu
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
          "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="🛰️ Uydu Görüntüsü"
).add_to(m)

print(f"✓ Base map: center=({center_lat:.4f}, {center_lon:.4f}), zoom=16")


✓ Base map: center=(41.0050, 28.9774), zoom=16


## 🏠 KATMAN 2: Bina İşaretleri

Her bina hasar sınıfına göre renkli CircleMarker, tıklanınca detaylı popup.


In [4]:
# ============================================================
# HÜCRE 8: Bina işaretleri + popup
# ============================================================
DMG_COLORS = {"no": "#2ecc71", "minor": "#f1c40f", "major": "#e67e22",
              "destroyed": "#e74c3c", "uncls": "#9b59b6"}
DMG_LABELS = {"no": "Hasarsız", "minor": "Hafif Hasar", "major": "Orta Hasar",
              "destroyed": "Ağır Hasar", "uncls": "Belirsiz"}

max_pri = popup_df["priority_score"].max()
if max_pri == 0: max_pri = 1

bldg_fg = folium.FeatureGroup(name="🏠 Hasarlı Binalar", show=True)

for _, b in popup_df.iterrows():
    cls = b["damage_class_name"]
    color = DMG_COLORS.get(cls, "#888")
    radius = 5 + 15 * (b["priority_score"] / max_pri)
    eta = b.get("eta_vehicle_min", None)
    eta_str = f"{eta:.1f} dk" if pd.notna(eta) else "—"
    n_alt = int(b.get("alternative_count", 0)) if pd.notna(b.get("alternative_count")) else 0
    method = b.get("route_method", "—")
    if pd.isna(method): method = "—"

    popup_html = f"""
    <div style="font-family:'Segoe UI',Arial; font-size:13px; min-width:210px;">
      <b style="font-size:15px;">🏠 Bina #{int(b['building_id'])}</b><br>
      <hr style="margin:4px 0;">
      <b>Hasar:</b> {DMG_LABELS.get(cls, cls)} <span style="color:{color};">●</span><br>
      <b>Alan:</b> {b['area_m2']:.0f} m²<br>
      <b>Tah. Nüfus:</b> {b['estimated_population']:.0f} kişi<br>
      <b>Öncelik:</b> {b['priority_score']:.1f}<br>
      <b>Hayatta Kalma:</b> %{b['survival_prob']*100:.0f}<br>
      <hr style="margin:4px 0;">
      <b>Ekip:</b> {int(b['team_id'])}<br>
      <b>Hastane:</b> {b['assigned_hospital_name']}<br>
      <b>Mesafe:</b> {b['distance_to_hospital_m']:.0f} m<br>
      <b>ETA (araç):</b> {eta_str}<br>
      <b>Rota:</b> {method}<br>
      <b>Alternatif:</b> {n_alt} rota
    </div>"""

    folium.CircleMarker(
        location=[b["lat_center"], b["lon_center"]],
        radius=radius, color=color, fill=True, fill_color=color,
        fill_opacity=0.8, weight=1.5,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"Bina #{int(b['building_id'])} — {DMG_LABELS.get(cls, cls)}",
    ).add_to(bldg_fg)

bldg_fg.add_to(m)
print(f"✓ Katman 2: {len(popup_df)} bina eklendi")


✓ Katman 2: 27 bina eklendi


## 🔥 KATMAN 3: Hasar Yoğunluğu

İki yöntem birden — kullanıcı LayerControl'den seçsin:
- **MarkerCluster** — deterministik, bilimsel
- **HeatMap** — KDE tabanlı, görsel etkili


In [5]:
# ============================================================
# HÜCRE 10: MarkerCluster + HeatMap
# ============================================================
from folium.plugins import MarkerCluster, HeatMap

# a) MarkerCluster (sadece hasarlı)
cluster_fg = folium.FeatureGroup(name="📊 Hasar Kümeleri", show=False)
mc = MarkerCluster()
damaged = popup_df[popup_df["damage_class_name"] != "no"]
for _, b in damaged.iterrows():
    cls = b["damage_class_name"]
    folium.Marker(
        location=[b["lat_center"], b["lon_center"]],
        icon=folium.Icon(
            color="red" if cls=="destroyed" else "orange" if cls=="major"
            else "green" if cls=="minor" else "purple",
            icon="exclamation-triangle" if cls in ("destroyed","major") else "info-sign"
        ),
        tooltip=f"#{int(b['building_id'])} {DMG_LABELS.get(cls,cls)} P={b['priority_score']:.0f}",
    ).add_to(mc)
mc.add_to(cluster_fg)
cluster_fg.add_to(m)

# b) HeatMap
heat_fg = folium.FeatureGroup(name="🌡️ Isı Haritası", show=False)
heat_data = [[b["lat_center"], b["lon_center"], b["priority_score"]]
             for _, b in damaged.iterrows() if b["priority_score"] > 0]
if heat_data:
    HeatMap(heat_data, radius=25, blur=15, max_zoom=17,
            gradient={0.4: "blue", 0.65: "lime", 1.0: "red"}
    ).add_to(heat_fg)
heat_fg.add_to(m)

print(f"✓ Katman 3: {len(damaged)} hasarlı bina (MarkerCluster + HeatMap)")


✓ Katman 3: 9 hasarlı bina (MarkerCluster + HeatMap)


## 👥 KATMAN 4: Ekip Sorumluluk Bölgeleri + Hastaneler


In [6]:
# ============================================================
# HÜCRE 12: Ekip bölgeleri + hastane markerları
# ============================================================
from scipy.spatial import Voronoi
import numpy as np

team_fg = folium.FeatureGroup(name="👥 Ekip Bölgeleri", show=True)

TEAM_COLORS = ["#e74c3c", "#3498db", "#2ecc71", "#e67e22", "#9b59b6",
               "#1abc9c", "#f39c12", "#34495e"]

# Voronoi bölgeleri — ekip merkezlerinden hesapla
centers = np.array([[t["lon"], t["lat"]] for t in TEAMS])
if len(centers) >= 4:
    vor = Voronoi(centers)
    # Finite Voronoi regions çiz
    for tid, region_idx in enumerate(vor.point_region):
        region = vor.regions[region_idx]
        if -1 in region or len(region) == 0:
            continue
        polygon = [(vor.vertices[i][1], vor.vertices[i][0]) for i in region]
        color = TEAM_COLORS[tid % len(TEAM_COLORS)]
        chain = team_chains.get(str(tid), {})
        n_bldg = chain.get("n_buildings", TEAMS[tid]["n_buildings"])
        dist = chain.get("tsp_distance_m", 0)
        folium.Polygon(
            locations=polygon,
            color=color, weight=2, fill=True,
            fill_color=color, fill_opacity=0.10, opacity=0.5,
            tooltip=f"Ekip {tid} — {n_bldg} bina, {dist:.0f}m rota",
        ).add_to(team_fg)

# Ekip merkezleri (X)
for t in TEAMS:
    tid = t["team_id"]
    chain = team_chains.get(str(tid), {})
    note = chain.get("note", "")
    folium.Marker(
        location=[t["lat"], t["lon"]],
        icon=folium.DivIcon(html=f'<div style="font-size:16px; font-weight:bold; '
             f'color:{TEAM_COLORS[tid % len(TEAM_COLORS)]};">✕</div>'),
        tooltip=f"Ekip {tid} merkezi · {note[:40]}",
    ).add_to(team_fg)

# Hastaneler
for h in HOSPITALS:
    # Kaç bina bu hastaneye atanmış?
    n_assigned = len(priority_df[priority_df["assigned_hospital_name"] == h["name"]])
    folium.Marker(
        location=[h["lat"], h["lon"]],
        icon=folium.Icon(color="green", icon="plus-sign"),
        popup=folium.Popup(
            f"<b>🏥 {h['name']}</b><br>{n_assigned} bina atanmış", max_width=250),
        tooltip=h["name"][:30],
    ).add_to(team_fg)

team_fg.add_to(m)
print(f"✓ Katman 4: {len(TEAMS)} ekip + {len(HOSPITALS)} hastane")


✓ Katman 4: 5 ekip + 5 hastane


## 🛣️ KATMAN 5: Yol Durumu (Gradient Edge Weight)

**Performans notu:** Binlerce edge var — sadece hasarlı olanları çiziyoruz.


In [7]:
# ============================================================
# HÜCRE 14: Gradient edge'ler (sadece hasarlı)
# ============================================================
road_fg = folium.FeatureGroup(name="🛣️ Yol Durumu", show=False)

n_drawn = 0
if G is not None:
    for u, v, data in G.edges(data=True):
        length = data.get("length", 0)
        gw = data.get("gradient_weight", length)
        blocked = data.get("blocked", False)

        if blocked or gw == float("inf"):
            color, weight, dash = "#e74c3c", 4, "10,6"
            label = "Kapalı Yol"
        elif length > 0 and gw > length * 2.5:
            color, weight, dash = "#e67e22", 3, None
            label = f"Tehlikeli (×{gw/length:.1f})"
        elif length > 0 and gw > length * 1.2:
            color, weight, dash = "#f1c40f", 2, None
            label = f"Yavaş (×{gw/length:.1f})"
        else:
            continue  # Normal yolları atla — performans

        coords = [(G.nodes[u]["y"], G.nodes[u]["x"]),
                  (G.nodes[v]["y"], G.nodes[v]["x"])]
        folium.PolyLine(
            coords, color=color, weight=weight,
            dash_array=dash, tooltip=label, opacity=0.85,
        ).add_to(road_fg)
        n_drawn += 1

road_fg.add_to(m)
print(f"✓ Katman 5: {n_drawn} hasarlı yol (geri kalanı atlandı — performans)")


✓ Katman 5: 0 hasarlı yol (geri kalanı atlandı — performans)


## 🗺️ KATMAN 6: Ekip Rotaları + Alternatif Rotalar


In [8]:
# ============================================================
# HÜCRE 16: Ekip rotaları (AntPath) + alternatif rotalar
# ============================================================
route_fg = folium.FeatureGroup(name="🗺️ Ekip Rotaları", show=True)

# AntPath import
try:
    from folium.plugins import AntPath
    has_antpath = True
except ImportError:
    has_antpath = False
    print("⚠️  AntPath yok, düz PolyLine kullanılacak")

for feat in team_routes_geo["features"]:
    coords_raw = feat["geometry"]["coordinates"]  # [lon, lat]
    if len(coords_raw) < 2:
        continue
    coords = [[c[1], c[0]] for c in coords_raw]  # [lat, lon]
    props = feat["properties"]
    tid = props.get("team_id", 0)
    color = props.get("color", TEAM_COLORS[tid % len(TEAM_COLORS)])
    dist = props.get("total_distance_m", 0)
    n_bldg = props.get("n_buildings", 0)
    hosp = props.get("hospital", "")
    note = props.get("note", "")

    tooltip = f"Ekip {tid}: {n_bldg} bina, {dist:.0f}m"
    popup_txt = (f"<b>Ekip {tid}</b><br>{n_bldg} bina · {dist:.0f}m<br>"
                 f"Hastane: {hosp[:25]}<br>{note[:50]}")

    if has_antpath:
        AntPath(coords, color=color, weight=4, delay=1000,
                dash_array=[20, 30], tooltip=tooltip,
                popup=folium.Popup(popup_txt, max_width=250)
        ).add_to(route_fg)
    else:
        folium.PolyLine(coords, color=color, weight=3,
                         tooltip=tooltip,
                         popup=folium.Popup(popup_txt, max_width=250)
        ).add_to(route_fg)

# Alternatif rotalar (ince kesik)
alt_fg = folium.FeatureGroup(name="🔀 Alternatif Rotalar", show=False)
n_alt_drawn = 0
for feat in alt_routes_geo["features"]:
    coords_raw = feat["geometry"]["coordinates"]
    if len(coords_raw) < 2:
        continue  # tek nokta = rota bulunamadı, atla
    coords = [[c[1], c[0]] for c in coords_raw]
    props = feat["properties"]
    bid = props.get("building_id", "?")
    rank = props.get("rank", 0)
    dist = props.get("physical_m", props.get("distance_m", 0))
    folium.PolyLine(
        coords, color="#888888", weight=2, dash_array="8,4",
        opacity=0.5,
        tooltip=f"Alt Rota {rank}: Bina #{bid}, {dist:.0f}m",
    ).add_to(alt_fg)
    n_alt_drawn += 1

alt_fg.add_to(m)
route_fg.add_to(m)
print(f"✓ Katman 6: {len(team_routes_geo['features'])} ekip rotası + {n_alt_drawn} alternatif")


✓ Katman 6: 5 ekip rotası + 9 alternatif


## 🚁 KATMAN 7: Helikopter İniş Noktaları

Tüm kara yolları açık, helikopter gerekmedi. Ama **potansiyel LZ'leri** gösteriyoruz (yollar kapanırsa hazır).


In [9]:
# ============================================================
# HÜCRE 18: LZ adayları (potansiyel)
# ============================================================
lz_fg = folium.FeatureGroup(name="🚁 Helikopter LZ", show=False)

if len(lz_df) > 0:
    # Top 5 LZ adayı göster
    top_lz = lz_df.head(5) if len(lz_df) >= 5 else lz_df
    for _, lz in top_lz.iterrows():
        name = str(lz.get("name", "LZ"))
        area = lz.get("approx_area_m2", 0)
        popup_html = (f"<b>🚁 {name}</b><br>"
                      f"Alan: {area:.0f} m²<br>"
                      f"Durum: Potansiyel<br>"
                      f"<i>Kara yolları şu an açık</i>")
        folium.Marker(
            location=[lz["lat"], lz["lon"]],
            icon=folium.Icon(color="darkblue", icon="plane", prefix="fa"),
            popup=folium.Popup(popup_html, max_width=230),
            tooltip=f"LZ: {name[:25]}",
        ).add_to(lz_fg)
    print(f"✓ Katman 7: {len(top_lz)} potansiyel LZ")
else:
    # lz_candidates boşsa not ekle
    folium.Marker(
        location=[center_lat, center_lon - 0.003],
        icon=folium.Icon(color="gray", icon="info-sign"),
        tooltip="Helikopter LZ: Kara yolları açık, LZ gerekmedi",
    ).add_to(lz_fg)
    print("✓ Katman 7: LZ gerekmedi (kara yolları açık)")

lz_fg.add_to(m)


✓ Katman 7: 5 potansiyel LZ


## ✨ Final Touches — LayerControl, Legend, Başlık


In [10]:
# ============================================================
# HÜCRE 20: LayerControl + plugins + overlay
# ============================================================

# LayerControl
folium.LayerControl(collapsed=False).add_to(m)

# Fullscreen
folium.plugins.Fullscreen(position="topright").add_to(m)

# MiniMap
folium.plugins.MiniMap(toggle_display=True, position="bottomleft").add_to(m)

# MeasureControl
folium.plugins.MeasureControl(position="topleft",
                               primary_length_unit="meters").add_to(m)

# İstatistikler
n_destroyed = int((priority_df["damage_class_name"] == "destroyed").sum())
n_major = int((priority_df["damage_class_name"] == "major").sum())
n_routes = len(team_routes_geo["features"])

# Başlık overlay
_n_bina = len(priority_df)
_n_ekip = len(TEAMS)
title_html = (
    '<div style="position:fixed; top:10px; left:60px; z-index:9999;'
    ' background:rgba(255,255,255,0.92); padding:14px 22px;'
    ' border-radius:10px; border:2px solid #e74c3c;'
    " font-family:'Segoe UI',Arial,sans-serif;"
    ' box-shadow:0 4px 12px rgba(0,0,0,0.15);">'
    ' <h3 style="margin:0; color:#e74c3c; font-size:18px;">'
    '   🚨 AFETSONAR</h3>'
    ' <p style="margin:3px 0 0; font-size:11px; color:#555;">'
    '   Calamitas AI · Afet Karar Destek Sistemi</p>'
    ' <p style="margin:2px 0 0; font-size:10px; color:#888;">'
    f'   {_n_bina} bina · {_n_ekip} ekip · {n_destroyed} yıkık · {n_routes} rota</p>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(title_html))

# Legend
legend_html = (
    '<div style="position:fixed; bottom:30px; right:10px; z-index:9999;'
    ' background:rgba(255,255,255,0.92); padding:12px 16px;'
    ' border-radius:8px; border:1px solid #ccc;'
    ' font-family:Arial; font-size:12px;'
    ' box-shadow:0 2px 6px rgba(0,0,0,0.15);">'
    ' <b style="font-size:13px;">Hasar Sınıfları</b><br>'
    ' <span style="color:#2ecc71;">●</span> Hasarsız<br>'
    ' <span style="color:#f1c40f;">●</span> Hafif Hasar<br>'
    ' <span style="color:#e67e22;">●</span> Orta Hasar<br>'
    ' <span style="color:#e74c3c;">●</span> Ağır Hasar / Yıkık<br>'
    ' <span style="color:#9b59b6;">●</span> Sınıflandırılmamış<br>'
    ' <hr style="margin:4px 0;">'
    ' <b>Diğer</b><br>'
    ' 🏥 Hastane &nbsp; 🚁 İniş Noktası<br>'
    ' <span style="color:#3498db;">━━</span> Kara Rotası<br>'
    ' <span style="color:#e74c3c;">╌╌</span> Kapalı Yol'
    '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))

print(f"✓ Final touches eklendi")
print(f"   Başlık: AFETSONAR · {len(priority_df)} bina · {n_destroyed} yıkık · {n_routes} rota")


✓ Final touches eklendi
   Başlık: AFETSONAR · 27 bina · 5 yıkık · 5 rota


## 💾 Kaydet ve Göster


In [11]:
# ============================================================
# HÜCRE 22: Kaydet + preview
# ============================================================
save_path = os.path.join(OUT_NB7, "afetsonar_master_map.html")
m.save(save_path)
file_kb = os.path.getsize(save_path) / 1024

print(f"✅ Master map kaydedildi: {save_path}")
print(f"   Boyut: {file_kb:.0f} KB ({file_kb/1024:.1f} MB)")
print(f"   {'✅ < 2 MB' if file_kb < 2048 else '⚠️  > 2 MB'}")

# === 7 Katman Doğrulama ===
print(f"\n{'='*50}")
print("🔍 7 KATMAN DOĞRULAMA")
print(f"{'='*50}")
katmanlar = [
    ("Base Tile", True, "OSM + Esri Uydu"),
    ("Bina İşaretleri", len(priority_df) > 0, f"{len(priority_df)} bina"),
    ("Hasar Yoğunluğu", True, "MarkerCluster + HeatMap"),
    ("Ekip Bölgeleri", len(TEAMS) > 0, f"{len(TEAMS)} ekip + {len(HOSPITALS)} hastane"),
    ("Yol Durumu", G is not None, f"{n_drawn if G else 0} hasarlı edge"),
    ("Ekip Rotaları", len(team_routes_geo["features"]) > 0,
     f"{len(team_routes_geo['features'])} rota + {n_alt_drawn} alt"),
    ("Helikopter LZ", True, f"{len(lz_df)} LZ adayı"),
]
for i, (name, ok, detail) in enumerate(katmanlar, 1):
    status = "✅" if ok else "❌"
    print(f"  {status} Katman {i}: {name} — {detail}")

print(f"\n🎉 Harita hazır! Tarayıcıda aç:")
print(f"   {save_path}")

# IFrame preview
try:
    from IPython.display import IFrame, display
    display(IFrame(save_path, width=1000, height=600))
except Exception:
    print("ℹ️  IFrame preview Colab'da çalışır")


✅ Master map kaydedildi: /content/drive/MyDrive/AFETSONAR/outputs/notebook7/afetsonar_master_map.html
   Boyut: 109 KB (0.1 MB)
   ✅ < 2 MB

🔍 7 KATMAN DOĞRULAMA
  ✅ Katman 1: Base Tile — OSM + Esri Uydu
  ✅ Katman 2: Bina İşaretleri — 27 bina
  ✅ Katman 3: Hasar Yoğunluğu — MarkerCluster + HeatMap
  ✅ Katman 4: Ekip Bölgeleri — 5 ekip + 5 hastane
  ✅ Katman 5: Yol Durumu — 0 hasarlı edge
  ✅ Katman 6: Ekip Rotaları — 5 rota + 9 alt
  ✅ Katman 7: Helikopter LZ — 68 LZ adayı

🎉 Harita hazır! Tarayıcıda aç:
   /content/drive/MyDrive/AFETSONAR/outputs/notebook7/afetsonar_master_map.html


## 📤 Bu Haritayı Nasıl Paylaşırsın?

| Yöntem | Nasıl |
|---|---|
| **Google Drive** | `afetsonar_master_map.html`'i Drive'dan paylaş |
| **WhatsApp** | Drive link'ini gönder |
| **Mobil** | HTML dosyasını Chrome'da aç (responsive) |
| **GitHub Pages** | Repo'ya push → Settings → Pages aç |
| **Sunum** | Screenshot al → PowerPoint'e koy |

> 💡 HTML dosyası ~200 KB — mobil uyumlu, offline çalışır (tile cache hariç).


## 🎓 Proje Özeti — AFETSONAR Pipeline

```
Drone/Uydu Görüntüsü
    ↓
[NB3] Student Model (4.5M param, 24ms, mIoU 0.43)
    ↓
[NB5] Hasar Tespiti → Priority → FEMA Survival → K-means Voronoi
    ↓
[NB6] Gradient Edge Weight → Multi-Target A* → TSP → k-Shortest → ETA
    ↓
[NB7] 7 Katmanlı İnteraktif Folium Harita
    ↓
AFAD / UMKE / AKUT → Arama-Kurtarma Karar Destek
```

### Bilimsel Temeller (20 kaynak)

Segmentation: SegFormer (Xie et al. 2021) + Knowledge Distillation (Hinton et al. 2015)
Damage: xBD (Gupta et al. 2019) + FEMA Survival Curve (Macintyre 2006)
Routing: A* (Hart et al. 1968) + Haversine (Sinnott 1984) + TSP-NN (Rosenkrantz 1977)
Clustering: K-means (MacQueen 1967) + Voronoi (Aurenhammer 1991)
Standards: NATO STANAG 3204, Türkiye Bina Yönetmeliği 2018, AFAD HHTK 2019

---
**Calamitas AI — Teknofest 2025 — Afet ve Acil Durum Teknolojileri — #811821**
